# Provider Data Cleaning Demo

This notebook demonstrates a reproducible cleaning workflow on deliberately messy **synthetic** provider data. Run `python make_dirty_data.py` from the repository root first to create `data_raw/providers_dirty.csv`.


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW = ROOT / 'data_raw' / 'providers_dirty.csv'
OUT = ROOT / 'data' / 'providers_clean.csv'

df = pd.read_csv(RAW, dtype=str)
df.head()


## 1. Profile the issues
We inspect row counts, missingness, duplicates, and representative categorical values before changing anything.


In [ ]:
print('Rows:', len(df))
print('Exact duplicate rows:', df.duplicated().sum())
display(df.isna().sum().sort_values(ascending=False).to_frame('missing'))
display(df[['city', 'state', 'agreement_status']].head(12))


## 2. Standardize text fields
Trim whitespace and normalize city, state, agreement status, and service delimiters.


In [ ]:
STATE_MAP = {
    'INDIANA': 'IN', 'OHIO': 'OH', 'KENTUCKY': 'KY',
    'ILLINOIS': 'IL', 'MICHIGAN': 'MI', 'WISCONSIN': 'WI',
}

for col in ['provider_id', 'provider_name', 'city', 'state', 'services', 'agreement_status']:
    df[col] = df[col].fillna('').str.strip()

df['city'] = df['city'].str.title()
df['state'] = df['state'].str.upper().replace(STATE_MAP)
df['agreement_status'] = df['agreement_status'].str.title()
df['services'] = (
    df['services']
    .str.replace(r'\s*[,/]\s*', ' | ', regex=True)
    .str.replace(r'\s*\|\s*', ' | ', regex=True)
)


## 3. Repair numeric fields and ZIP codes
Convert values such as `4.1 stars` and `43.8%` back to numeric types, while preserving missing values.


In [ ]:
df['rating'] = pd.to_numeric(
    df['rating'].fillna('').str.extract(r'([0-9]+(?:\.[0-9]+)?)', expand=False),
    errors='coerce',
)
df['utilization_pct'] = pd.to_numeric(
    df['utilization_pct'].fillna('').str.extract(r'([0-9]+(?:\.[0-9]+)?)', expand=False),
    errors='coerce',
)

def clean_zip(value):
    digits = ''.join(ch for ch in str(value) if ch.isdigit())
    return digits[:5].zfill(5) if digits else pd.NA

df['zip_code'] = df['zip_code'].apply(clean_zip)
for col in ['latitude', 'longitude', 'capacity', 'active_jobs', 'average_response_hours']:
    df[col] = pd.to_numeric(df[col], errors='coerce')


## 4. Remove unusable and duplicate records
Drop the fully blank row, remove exact duplicates, and require the minimum fields needed for provider analysis.


In [ ]:
df = df.replace('', pd.NA)
df = df.dropna(how='all').drop_duplicates()
df = df.drop_duplicates(subset=['provider_id'], keep='first')
df = df.dropna(subset=['provider_id', 'provider_name', 'state', 'latitude', 'longitude'])
df = df.reset_index(drop=True)
print('Clean rows:', len(df))


## 5. Validate the cleaned table
These assertions make the cleaning rules explicit and fail loudly if the pipeline stops producing analysis-ready data.


In [ ]:
assert df['provider_id'].is_unique
assert df['provider_name'].str.strip().eq(df['provider_name']).all()
assert df['state'].isin(['IN', 'OH', 'KY', 'IL', 'MI', 'WI']).all()
assert df['rating'].dropna().between(1, 5).all()
assert df['utilization_pct'].dropna().ge(0).all()
assert df['zip_code'].dropna().str.fullmatch(r'\d{5}').all()
assert df[['latitude', 'longitude']].notna().all().all()
print('Validation passed.')


## 6. Save the analysis-ready result


In [ ]:
df.to_csv(OUT, index=False)
print(f'Saved {len(df):,} rows to {OUT}')
